# 第 24 章 · profiler：给训练拍一段"录像"

本章回答：`torch.profiler` 怎么用？输出的表格里 Self 和 Total 有什么区别？时间线（trace）上该找什么？用一次真实的测量，看"改一处、再测"怎么走一遍。

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/24-profiler.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch13「使用 PyTorch 分析器识别瓶颈」（`ch13.md` 第 131–291 行）、「Nsight Systems 与 NVTX 时间线」（第 292–380 行）、NVTX 标记（第 19–130 行）；官方教程「PyTorch Profiler」（〔在线〕）；本章表格和数字是本机 Mac CPU 上 torch 2.15.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
from torch.profiler import ProfilerActivity, profile, record_function, schedule
import torch
import torch.nn as nn

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 读表 | Self 前几名 | `profile_step(False)` 的 top | mm、addmm、mul_、addcdiv_、lerp_ |
| 第三步 改一处 | fused AdamW | `profile_step(False / True)` | 每步约 2.5 → 1.8 ms，优化器约 1.2 → 0.5 ms |
| 附录 Amdahl | 1 / ((1−p) + p/s) | `amdahl(0.479, 2.45)` | 1.40 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `amdahl`**（来自 `ch24_profiler.py`）：下面要用到的小工具。

In [ ]:
def amdahl(p: float, s: float) -> float:
    return 1 / ((1 - p) + p / s)

**函数 `make`**（来自 `ch24_profiler.py`）：下面要用到的小工具。

In [ ]:
def make(fused: bool):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(512, 2048), nn.GELU(), nn.Linear(2048, 512), nn.LayerNorm(512))
    return m, torch.optim.AdamW(m.parameters(), fused=fused), torch.randn(64, 512)

**函数 `profile_step`**（来自 `ch24_profiler.py`）：返回 (每步总 ms, 每步优化器 ms, 优化器占比, 按 Self CPU 排的前 5 个算子名)。

In [ ]:
def profile_step(fused: bool, active: int = 10):
    """返回 (每步总 ms, 每步优化器 ms, 优化器占比, 按 Self CPU 排的前 5 个算子名)。"""
    m, opt, x = make(fused)

    def step():
        with record_function("train_step"):
            loss = m(x).pow(2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

    with profile(activities=[ProfilerActivity.CPU], schedule=schedule(wait=1, warmup=2, active=active)) as prof:
        for _ in range(3 + active):
            step()
            prof.step()
    ka = prof.key_averages()
    total = sum(e.self_cpu_time_total for e in ka)
    opt_ev = next(e for e in ka if e.key.startswith("Optimizer.step"))
    top = [e.key for e in sorted(ka, key=lambda e: -e.self_cpu_time_total)[:5]]
    return total / active / 1e3, opt_ev.cpu_time_total / active / 1e3, opt_ev.cpu_time_total / total, top

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
for fused in (False, True):
    t, o, share, top = profile_step(fused)
    print(f"\nfused={fused}: 每步 {t:.2f} ms，优化器 {o:.2f} ms（{share:.0%}）；Self 前 5：{top}")
print("\nAmdahl：占 47.9% 的部分快 2.45 倍 → 总加速", round(amdahl(0.479, 2.45), 2))

## 3. 代码里才看得到的

1. **每次测的数略有不同**：讲义里是 2.61 / 1.82 ms，测试只核对"fused 更快"。
2. **fused 后 top 里出现 `aten::_fused_adamw_`**：一个 kernel 顶替了一串小算子。

## 4. 自己改一改

- 把 batch 从 64 改成 1024，看优化器占比怎么变。
- 用 `prof.export_chrome_trace("t.json")` 导出，拖进 ui.perfetto.dev 看。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> profile 加 schedule，跳过热身只录稳定步；record_function 起名字，trace 拖进 perfetto；
> Self 是自己干、Total 含子孙，Total 大 Self 小底下全是碎活；
> 找最大的块改一处再测；时间线上找空白、找细条、找没藏住的 NCCL。